# Synthetic bank-disclosure workflow

This notebook demonstrates 252 generated section records, correlated mock sentiment, fixed mock entities and random mock topics. Optional real-model adapters are implemented but were not executed for the published sample. No real filing language, investment recommendation or predictive warning is established. See [REPORT.md](../REPORT.md) for the current interpretation.

Edited code cells are unexecuted until rerun; unchanged numerical outputs remain saved evidence.


## Setup

The default path is synthetic. FinBERT and BERTopic switches do not repair the unfinished EDGAR date, section and schema contract. Do not use the real-data branch as an end-to-end empirical workflow.


In [11]:
import sys, yaml, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.nlp_ei.synthetic import generate_synthetic_corpus
from src.nlp_ei.sentiment import run_sentiment
from src.nlp_ei.ner import extract_entities, top_entities_by_type
from src.nlp_ei.topics import fit_bertopic, mock_topics
from src.nlp_ei.timeseries import (
    aggregate_quarterly, aggregate_quarterly_by_section,
    compute_section_gap, flag_sentiment_drops, pivot_heatmap,
    disaggregate_by_tier, survivorship_analysis,
)
from src.nlp_ei.visualise import (
    plot_sentiment_heatmap, plot_sentiment_trends, plot_entity_freq,
    plot_section_gap, plot_tier_comparison, plot_score_distribution,
    plot_survivorship,
)

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

cfg = yaml.safe_load((ROOT / "config" / "default.yaml").read_text())
edgar_cfg = cfg["edgar"]
nlp_cfg = cfg["nlp"]
anomaly_cfg = cfg["anomaly"]
scenario_cfg = cfg["scenario"]

FAILED_BANKS = scenario_cfg["failed_banks"]
GAP_ALERT    = anomaly_cfg["section_gap_alert_threshold"]
ZSCORE_THRESH = anomaly_cfg["sentiment_zscore_threshold"]

# Tier colour map for consistent styling
TIER_COLOURS = {"gsib": "#2196F3", "super_regional": "#4CAF50", "stressed": "#F44336"}

sns.set_theme(style="whitegrid")
print("Config loaded. Scenario:", scenario_cfg["name"])
print("Tickers:", edgar_cfg["tickers"])
print("Date range:", edgar_cfg["date_range"]["start"], "–", edgar_cfg["date_range"]["end"])
print("Failed banks:", FAILED_BANKS)

Config loaded. Scenario: us_regional_bank_stress_2020_2024
Tickers: ['JPM', 'BAC', 'USB', 'RF', 'KEY', 'ZION', 'SIVB', 'FRC']
Date range: 2020-01-01 – 2024-03-31
Failed banks: ['SIVB', 'FRC']


## Scenario assumptions

Bank labels, tier health, regime terms and terminal stress are hand-assigned generator settings informed by known outcomes. SIVB and FRC share a programmed 2023 Q1 cutoff; the real closures occurred in March and May 2023. Generated filing dates are artificial quarter ends. The scenario cannot establish warning lead time.


## Corpus assembly

Verify generated section counts, quarter coverage and programmed cutoffs. The 252 rows are generated sections across 126 ticker-quarter cells, not authentic SEC filings.


In [12]:
USE_SYNTHETIC = cfg["synthetic"]["use_synthetic"]
USE_FINBERT   = False    # set True if transformers + torch installed
USE_BERTOPIC  = False    # set True if bertopic + sentence-transformers installed

if USE_SYNTHETIC:
    corpus = generate_synthetic_corpus(
        tickers=edgar_cfg["tickers"],
        start_date=edgar_cfg["date_range"]["start"],
        end_date=edgar_cfg["date_range"]["end"],
        seed=42,
    )
else:
    from src.nlp_ei.downloader import download_filings, load_filing_texts
    print("Downloading from EDGAR ...")
    download_filings(
        tickers=edgar_cfg["tickers"],
        form_types=edgar_cfg["form_types"],
        start_date=edgar_cfg["date_range"]["start"],
        end_date=edgar_cfg["date_range"]["end"],
    )
    corpus = load_filing_texts("data/filings", edgar_cfg["tickers"])

print(f"Corpus: {len(corpus):,} documents | {corpus['ticker'].nunique()} tickers | "
      f"{corpus['year_quarter'].nunique()} quarters")
corpus.head(3)

Corpus: 252 documents | 8 tickers | 17 quarters


,ticker,filing_date,quarter,year_quarter,section,text,true_sentiment,tier,balance_sheet_health,rate_regime,failed_bank
0,BAC,2020-03-31,2020Q1,2020Q1,risk_factors,Our risk models may underestimate losses durin...,0.043687,gsib,0.82,pre_covid,False
1,BAC,2020-03-31,2020Q1,2020Q1,mda,"Fee income rose 6% year-over-year, with wealth...",0.459235,gsib,0.82,pre_covid,False
2,BAC,2020-06-30,2020Q2,2020Q2,risk_factors,Elevated unemployment of 20% increases the ris...,-0.214529,gsib,0.82,covid_stress,False


In [ ]:
# Balance check: documents per ticker
doc_counts = corpus.groupby(["ticker", "section"]).size().unstack(fill_value=0)
doc_counts["total"] = doc_counts.sum(axis=1)
doc_counts["tier"] = corpus.drop_duplicates("ticker").set_index("ticker")["tier"]
doc_counts["failed"] = corpus.drop_duplicates("ticker").set_index("ticker")["failed_bank"]
print("Documents per ticker:")
print(doc_counts.sort_values("tier").to_string())
print()
print("Note: SIVB and FRC have fewer documents, their programmed records end in 2022Q4.")

In [14]:
# Rate-regime distribution
if "rate_regime" in corpus.columns:
    regime_counts = corpus.groupby("rate_regime")["ticker"].count().rename("n_docs")
    print("Documents per rate regime:")
    print(regime_counts.to_string())
    print()
    print("Quarter range by ticker:")
    q_range = corpus.groupby("ticker")["year_quarter"].agg(["min", "max"])
    print(q_range.to_string())

Documents per rate regime:
rate_regime
covid_stress    48
hiking          88
post_hike       36
pre_covid       16
recovery        64

Quarter range by ticker:
           min     max
ticker                
BAC     2020Q1  2024Q1
FRC     2020Q1  2022Q4
JPM     2020Q1  2024Q1
KEY     2020Q1  2024Q1
RF      2020Q1  2024Q1
SIVB    2020Q1  2022Q4
USB     2020Q1  2024Q1
ZION    2020Q1  2024Q1


## Mock sentiment and optional FinBERT

Published scores read planted true_sentiment plus noise rather than interpreting text. Saved correlation 0.9102 checks that constructed relationship. FinBERT is optional and unverified here; its adapter uses approximate character-based chunking and token truncation. No external accuracy benchmark validates these outputs.


In [15]:
print(f"Scoring with: {'FinBERT' if USE_FINBERT else 'mock scorer (correlated)'}")
sentiment_df = run_sentiment(
    corpus,
    use_finbert=USE_FINBERT,
    model_name=nlp_cfg["finbert_model"],
    batch_size=nlp_cfg["sentence_batch_size"],
)

# Verify the correlation is meaningful
if "true_sentiment" in sentiment_df.columns:
    corr = float(np.corrcoef(sentiment_df["true_sentiment"], sentiment_df["score"])[0, 1])
    print(f"Mock scorer <-> true_sentiment correlation: r = {corr:.3f}")

print(f"\nScored {len(sentiment_df):,} documents")
sentiment_df[["ticker", "year_quarter", "section", "positive", "negative", "neutral",
              "score", "label"]].head(6)

Scoring with: mock scorer (correlated)
Mock scorer <-> true_sentiment correlation: r = 0.910

Scored 252 documents


,ticker,year_quarter,section,positive,negative,neutral,score,label
0,BAC,2020Q1,risk_factors,0.4480,0.3715,0.1806,0.0765,positive
1,BAC,2020Q1,mda,0.6013,0.3077,0.0911,0.2936,positive
2,BAC,2020Q2,risk_factors,0.2040,0.6402,0.1558,-0.4362,negative
3,BAC,2020Q2,mda,0.4596,0.3060,0.2343,0.1536,positive
4,BAC,2020Q3,risk_factors,0.2425,0.5983,0.1592,-0.3557,negative
5,BAC,2020Q3,mda,0.5322,0.4178,0.0500,0.1144,positive


In [16]:
# Section-level means: MD&A is structurally more positive than risk factors
print("Mean net sentiment by section:")
print(sentiment_df.groupby("section")["score"].describe().round(3).to_string())
print()
print("Mean net sentiment by tier:")
if "tier" in sentiment_df.columns:
    print(sentiment_df.groupby("tier")["score"].agg(["mean","std","min","max"]).round(3).to_string())

Mean net sentiment by section:
              count   mean    std    min    25%    50%    75%    max
section                                                             
mda           126.0  0.202  0.157 -0.214  0.101  0.204  0.319  0.514
risk_factors  126.0 -0.177  0.210 -0.654 -0.314 -0.188 -0.024  0.505

Mean net sentiment by tier:
                 mean    std    min    max
tier                                      
gsib            0.114  0.237 -0.436  0.514
stressed       -0.052  0.274 -0.654  0.486
super_regional  0.021  0.247 -0.536  0.446


## Generated score distributions

Section and tier differences arise from planted parameters. These plots inspect mock distributions, not evidence about actual disclosure language or model discrimination.


In [17]:
fig = plot_score_distribution(sentiment_df, save_path=OUTPUTS / "score_distribution.png")
plt.show()
print("Saved score_distribution.png")

Saved score_distribution.png


In [18]:
# Quantitative summary of the distributional difference
for section in ["mda", "risk_factors"]:
    sub = sentiment_df[sentiment_df["section"] == section]["score"]
    print(f"{section:15s}: mean={sub.mean():.3f}  std={sub.std():.3f}  "
          f"p10={sub.quantile(0.10):.3f}  p90={sub.quantile(0.90):.3f}")

mda            : mean=0.202  std=0.157  p10=-0.008  p90=0.398
risk_factors   : mean=-0.177  std=0.210  p10=-0.437  p90=0.086


## Section-score gaps

Gap equals mock MD&A minus mock Risk Factors. The chosen 0.60 cutoff identifies scenario differences, not validated management bias. SIVB exceeds it in 2021 Q2, 2022 Q2 and 2022 Q3, with peak 0.9247 in 2021 Q2. KEY has the overall maximum, 0.9724 in 2022 Q4. Vertical orange markers identify gap-exceedance quarters.


In [19]:
# Section-level aggregation
section_quarterly = aggregate_quarterly_by_section(sentiment_df)
section_quarterly = compute_section_gap(section_quarterly)

# Add tier for downstream plotting
tier_map = corpus.drop_duplicates("ticker").set_index("ticker")["tier"].to_dict()
section_quarterly["tier"] = section_quarterly["ticker"].map(tier_map)

print("Section-level quarterly summary (first 6 rows):")
print(section_quarterly[["ticker", "year_quarter", "mda_sentiment",
                           "risk_sentiment", "section_gap"]].head(6).round(3).to_string(index=False))

Section-level quarterly summary (first 6 rows):
ticker year_quarter  mda_sentiment  risk_sentiment  section_gap
   BAC       2020Q1          0.294           0.076        0.217
   BAC       2020Q2          0.154          -0.436        0.590
   BAC       2020Q3          0.114          -0.356        0.470
   BAC       2020Q4         -0.065          -0.107        0.041
   BAC       2021Q1          0.365          -0.009        0.374
   BAC       2021Q2          0.476           0.221        0.256


In [20]:
# Tickers that exceed the gap-alert threshold
alert_rows = section_quarterly[section_quarterly["section_gap"] > GAP_ALERT]
print(f"Quarters exceeding gap-alert threshold (>{GAP_ALERT}):")
print(alert_rows[["ticker","year_quarter","mda_sentiment","risk_sentiment","section_gap"]]
      .sort_values("section_gap", ascending=False)
      .round(3)
      .to_string(index=False))
print()

# Max gap per ticker
print("Maximum section gap by ticker:")
gap_max = (
    section_quarterly.groupby("ticker")["section_gap"]
    .agg(["max","mean"])
    .round(3)
    .rename(columns={"max":"max_gap","mean":"mean_gap"})
)
print(gap_max.sort_values("max_gap", ascending=False).to_string())

Quarters exceeding gap-alert threshold (>0.6):
ticker year_quarter  mda_sentiment  risk_sentiment  section_gap
   KEY       2022Q4          0.318          -0.654        0.972
  SIVB       2021Q2          0.486          -0.439        0.925
  SIVB       2022Q2          0.449          -0.465        0.915
    RF       2022Q3          0.406          -0.478        0.884
  ZION       2022Q4          0.402          -0.408        0.810
    RF       2022Q4          0.319          -0.436        0.755
   BAC       2022Q2          0.352          -0.379        0.731
  SIVB       2022Q3          0.116          -0.600        0.716
  ZION       2022Q1          0.286          -0.416        0.702
   USB       2022Q2          0.408          -0.288        0.696
   KEY       2022Q2          0.394          -0.291        0.685
   JPM       2023Q1          0.329          -0.341        0.670
   KEY       2023Q1          0.349          -0.297        0.646
  ZION       2022Q3          0.092          -0.549       

In [ ]:
fig = plot_section_gap(section_quarterly, gap_alert=GAP_ALERT,
                       save_path=OUTPUTS / "section_gap.png")
plt.show()
print("Saved section_gap.png")
print()
print("Blue=MD&A sentiment  Red=Risk-Factors sentiment  Shaded=gap  Vertical orange lines=gap-exceedance quarters")
print("These are mock scores with planted section differences.")

## Full-series anomaly calculation

Z-scores use every supplied quarter for that ticker and sample standard deviation. Earlier values can depend on later quarters. The default threshold flags 11 ticker-quarter cells: eight programmed COVID-window flags across five tickers, two SIVB/FRC flags in 2022 Q4 and one RF flag in 2023 Q1. FRC also flags in 2020 Q3. USB in 2020 Q3 has the lowest z-score, -2.2599. These are retrospective mock deviations.


In [22]:
quarterly = aggregate_quarterly(sentiment_df)
quarterly["tier"] = quarterly["ticker"].map(tier_map)
anomaly_df = flag_sentiment_drops(quarterly, threshold_zscore=ZSCORE_THRESH)
anomaly_df["tier"] = anomaly_df["ticker"].map(tier_map)

n_anomalies = int(anomaly_df["anomaly"].sum())
print(f"Anomaly quarters detected (z < -{ZSCORE_THRESH}): {n_anomalies}")
print()
print("Flagged quarters:")
flagged = anomaly_df[anomaly_df["anomaly"]]
print(flagged[["ticker","year_quarter","avg_sentiment","z_score","tier"]]
      .sort_values("z_score")
      .round(3)
      .to_string(index=False))

Anomaly quarters detected (z < -1.5): 11

Flagged quarters:
ticker year_quarter  avg_sentiment  z_score           tier
   USB       2020Q3         -0.197   -2.260 super_regional
  ZION       2020Q3         -0.303   -1.989       stressed
  SIVB       2022Q4         -0.394   -1.905       stressed
   USB       2020Q4         -0.145   -1.820 super_regional
   KEY       2020Q2         -0.211   -1.776       stressed
    RF       2023Q1         -0.274   -1.715 super_regional
   FRC       2022Q4         -0.318   -1.711       stressed
   BAC       2020Q2         -0.141   -1.703           gsib
   KEY       2020Q3         -0.188   -1.577       stressed
   BAC       2020Q3         -0.121   -1.544           gsib
   FRC       2020Q3         -0.296   -1.540       stressed


In [23]:
# Focus on the pre-failure signals for SIVB and FRC
print("SIVB quarterly series:")
sivb = anomaly_df[anomaly_df["ticker"] == "SIVB"][
    ["year_quarter","avg_sentiment","z_score","anomaly","qoq_change"]
].round(3)
print(sivb.to_string(index=False))
print()
print("FRC quarterly series:")
frc = anomaly_df[anomaly_df["ticker"] == "FRC"][
    ["year_quarter","avg_sentiment","z_score","anomaly","qoq_change"]
].round(3)
print(frc.to_string(index=False))

SIVB quarterly series:
year_quarter  avg_sentiment  z_score  anomaly  qoq_change
      2020Q1         -0.156   -0.266    False         NaN
      2020Q2         -0.283   -1.140    False      -0.127
      2020Q3         -0.177   -0.407    False       0.106
      2020Q4         -0.115    0.019    False       0.062
      2021Q1          0.016    0.922    False       0.131
      2021Q2          0.024    0.977    False       0.008
      2021Q3         -0.050    0.465    False      -0.074
      2021Q4          0.111    1.576    False       0.161
      2022Q1         -0.138   -0.140    False      -0.249
      2022Q2         -0.008    0.757    False       0.130
      2022Q3         -0.242   -0.859    False      -0.234
      2022Q4         -0.394   -1.905     True      -0.152

FRC quarterly series:
year_quarter  avg_sentiment  z_score  anomaly  qoq_change
      2020Q1         -0.016    0.697    False         NaN
      2020Q2         -0.150   -0.372    False      -0.134
      2020Q3         -0.29

In [ ]:
hm = pivot_heatmap(quarterly)
fig = plot_sentiment_heatmap(hm, save_path=OUTPUTS / "sentiment_heatmap.png")
plt.show()
print("Saved sentiment_heatmap.png")
print()
print("Marker = programmed record cutoff. Red = negative raw mock net score, not z-score.")
print("The trajectories reflect programmed regime and health settings.")

In [25]:
fig = plot_sentiment_trends(quarterly, anomaly_df=anomaly_df,
                            save_path=OUTPUTS / "sentiment_trends.png")
plt.show()
print("Saved sentiment_trends.png")
print("Red dashed lines mark anomaly-flagged quarters.")

Saved sentiment_trends.png
Red dashed lines mark anomaly-flagged quarters.


## Assigned tiers

Fixed health and regime parameters create these differences. GSIB combined means are negative in 2020 Q2 through Q4 despite a full-period mean near +0.1144. No capital-buffer, depositor or interest-margin mechanism is measured.


In [26]:
tier_agg = disaggregate_by_tier(sentiment_df)
print("Quarterly sentiment by tier:")
print(tier_agg.pivot_table(
    index="tier", values="avg_sentiment", aggfunc=["mean","std","min"]
).round(3).to_string())
print()

# Section-level means by tier
print("Section-level means by tier:")
for tier in ["gsib", "super_regional", "stressed"]:
    tickers_in_tier = [t for t, v in tier_map.items() if v == tier]
    sub = sentiment_df[sentiment_df["ticker"].isin(tickers_in_tier)]
    m = sub.groupby("section")["score"].mean()
    mda_m = m.get("mda", float("nan"))
    risk_m = m.get("risk_factors", float("nan"))
    print(f"  {tier:20s}  MD&A = {mda_m:+.3f}  Risk = {risk_m:+.3f}  "
          f"Gap = {mda_m - risk_m:.3f}")

Quarterly sentiment by tier:
                        mean           std           min
               avg_sentiment avg_sentiment avg_sentiment
tier                                                    
gsib                   0.114         0.120        -0.087
stressed              -0.038         0.116        -0.241
super_regional         0.021         0.110        -0.181

Section-level means by tier:
  gsib                  MD&A = +0.278  Risk = -0.049  Gap = 0.328
  super_regional        MD&A = +0.203  Risk = -0.160  Gap = 0.363
  stressed              MD&A = +0.158  Risk = -0.262  Gap = 0.419


In [27]:
fig = plot_tier_comparison(tier_agg, save_path=OUTPUTS / "tier_comparison.png")
plt.show()
print("Saved tier_comparison.png")
print()
print("Blue=GSIB  Green=Super-Regional  Red=Stressed")
print("Shaded bands: grey=COVID stress, orange=rate hiking cycle")

Saved tier_comparison.png

Blue=GSIB  Green=Super-Regional  Red=Stressed
Shaded bands: grey=COVID stress, orange=rate hiking cycle


## Mock entity frequencies

The published fallback is a fixed entity table scaled by 252/120 = 2.1 and truncated to integer counts. No extracted mention trends, geographic concentration or entity co-occurrence was measured.


In [28]:
entity_df = extract_entities(corpus, spacy_model=nlp_cfg["spacy_model"])
print(f"Unique entity types: {sorted(entity_df['label'].unique())}")
print(f"Total entity rows: {len(entity_df):,}")
entity_df.head(15)

spaCy model 'en_core_web_sm' not found, returning banking mock entities
Unique entity types: ['GPE', 'MONEY', 'ORG', 'PERCENT', 'PRODUCT']
Total entity rows: 25


,entity,label,count
0,United States,GPE,655
1,Federal Reserve,ORG,596
2,commercial real estate,PRODUCT,426
3,California,GPE,392
4,held-to-maturity,PRODUCT,373
5,New York,GPE,346
6,FDIC,ORG,327
7,10%,PERCENT,298
8,Paycheck Protection Program,PRODUCT,281
9,5%,PERCENT,247


In [29]:
entity_df.to_csv(OUTPUTS / "entity_freq.csv", index=False)
fig = plot_entity_freq(entity_df, top_n=15, save_path=OUTPUTS / "entity_freq.png")
plt.show()
print("Saved entity_freq.csv and entity_freq.png")
print()
print("Top entities by type:")
print(top_entities_by_type(entity_df, n=4).to_string(index=False))

Saved entity_freq.csv and entity_freq.png

Top entities by type:
                                   entity  count
                            United States    655
                               California    392
                                 New York    346
                                    China    205
                               $1 billion    201
                             $500 million    176
                               $2 billion    149
                             $250 million    121
                          Federal Reserve    596
                                     FDIC    327
                   Federal Home Loan Bank    186
Office of the Comptroller of the Currency    134
                                      10%    298
                                       5%    247
                                     1.5%    186
                         525 basis points    130
                   commercial real estate    426
                         held-to-maturity    373
    

## Mock topic labels

Seven supplied labels are randomly assigned using fixed weights and seed 0. They ignore text content. The optional BERTopic adapter is distinct; a real-data run may discover different themes and requires separate evaluation.


In [30]:
texts = corpus["text"].tolist()

if USE_BERTOPIC:
    print("Fitting BERTopic model ...")
    topic_model, topics, probs = fit_bertopic(
        texts,
        min_topic_size=nlp_cfg["bertopic_min_topic_size"],
        n_gram_range=tuple(nlp_cfg["bertopic_n_gram_range"]),
    )
    topic_info = topic_model.get_topic_info()
    print(f"Topics found: {len(topic_info) - 1}  (topic -1 = outliers, expected 10–30%)")
    print(topic_info.head(12))

    try:
        fig_topics = topic_model.visualize_topics()
        fig_topics.write_html(str(OUTPUTS / "topic_map.html"))
        print("\nSaved topic_map.html (interactive UMAP)")
    except Exception as e:
        print(f"Could not save topic map: {e}")

    topic_df = pd.DataFrame({"text_idx": range(len(texts)), "topic_id": topics})
else:
    print("Using mock topics (set USE_BERTOPIC=True for real BERTopic)")
    topic_df = mock_topics(texts)
    print("\nMock topic distribution:")
    print(topic_df["topic_label"].value_counts().to_string())

Using mock topics (set USE_BERTOPIC=True for real BERTopic)

Mock topic distribution:
topic_label
deposits, liquidity, outflows, uninsured                   51
net interest margin, rate, asset-sensitive, repricing      44
held-to-maturity, unrealized loss, securities, duration    44
credit loss, provision, non-performing, charge-offs        37
commercial real estate, office, vacancy, refinancing       33
capital, CET1, regulatory, stress test                     25
technology sector, venture capital, startup, SVB           18


## In-sample threshold sensitivity

Threshold magnitudes 1, 1.25, 1.5, 1.75 and 2 give total flags 25, 16, 11, 5 and 1. Counts in all SIVB/FRC quarters are 4, 3, 3, 1 and 0. Those are not pre-failure hit counts or independent calibration. Event-window labels are assigned by ticker and quarter.


In [ ]:
# Classify anomaly flags
def classify_anomaly(row):
    q = str(row["year_quarter"])
    ticker = row["ticker"]
    if not row["anomaly"]:
        return "not_flagged"
    # Programmed ticker-quarter category, not a predictive hit
    if ticker in ["SIVB", "FRC"] and q in ["2022Q3", "2022Q4"]:
        return "late_2022_programmed_window"
    # Programmed COVID-window category
    if q in ["2020Q2", "2020Q3", "2020Q4"]:
        return "covid_stress"
    # Anything else
    return "other_flag"

anomaly_df["flag_type"] = anomaly_df.apply(classify_anomaly, axis=1)
flag_summary = anomaly_df[anomaly_df["anomaly"]]["flag_type"].value_counts()
print("Anomaly flag classification:")
print(flag_summary.to_string())
print()
total_flags = flag_summary.sum()
pre_failure = flag_summary.get("late_2022_programmed_window", 0)
covid = flag_summary.get("covid_stress", 0)
print(f"Programmed late-2022 window (SIVB/FRC 2022 Q3-Q4): {pre_failure}/{total_flags} flags")
print(f"Programmed COVID window: {covid}/{total_flags} flags")

In [32]:
# Threshold sensitivity: how many flags at different z-score thresholds?
print("Flag count by z-score threshold:")
print(f"{'Threshold':>12}  {'Total flags':>12}  {'SIVB/FRC flags':>14}")
for thresh in [1.0, 1.25, 1.5, 1.75, 2.0, 2.5]:
    tmp = flag_sentiment_drops(quarterly, threshold_zscore=thresh)
    n_total = tmp["anomaly"].sum()
    n_sivb_frc = tmp[tmp["ticker"].isin(["SIVB","FRC"]) & tmp["anomaly"]].shape[0]
    print(f"{thresh:>12.2f}  {n_total:>12d}  {n_sivb_frc:>14d}")
print()
print("Threshold 1.5 is the default. Lower = more flags (higher sensitivity).")

Flag count by z-score threshold:
   Threshold   Total flags  SIVB/FRC flags
        1.00            25               4
        1.25            16               3
        1.50            11               3
        1.75             5               1
        2.00             1               0
        2.50             0               0

Threshold 1.5 is the default. Lower = more flags (higher sensitivity).


## Retrospective cohort comparison

Available-record and survivor-only means of 17 quarterly averages are 0.0152679 and 0.0416132, a difference of 0.0263453 score units. The survivor cohort is selected using known outcomes. The curves coincide from 2023 Q1. There is no carry-forward, adjusted benchmark, constant-offset correction or attribution of later recovery.


In [ ]:
surv_df = survivorship_analysis(sentiment_df, failed_tickers=FAILED_BANKS)
print("Sector average: all banks vs. survivors only")
print(surv_df[["year_quarter","avg_sentiment_all","avg_sentiment_survivors",
               "survivorship_bias"]].round(3).to_string(index=False))

overall_all = surv_df["avg_sentiment_all"].mean()
overall_surv = surv_df["avg_sentiment_survivors"].mean()
print(f"\nOverall mean, all: {overall_all:.4f}  |  survivors only: {overall_surv:.4f}")
print(f"Retrospective cohort difference: {overall_surv - overall_all:+.6f} score units")

In [ ]:
fig = plot_survivorship(surv_df, save_path=OUTPUTS / "survivorship.png")
plt.show()
print("Saved survivorship.png")
print()
print("Grey line: sector average including all banks (incl. SIVB & FRC until 2022Q4)")
print("Green line: sector average from survivors only")
print("Orange shading: the survivorship bias (the gap between the two series)")
print("Red dotted line: programmed record cutoff (2023 Q1)")

## Engineering synthesis

Keep planted signals separate from measured mechanisms. No bank investment or exit decision follows from these outputs. Empirical work needs repaired authentic ingestion, recorded executed backends without fallback, complete token coverage and evaluation on unseen periods and entities.


In [ ]:
# ─── SIGNAL 1: Anomaly Detection ───────────────────────────────
sivb_flags = anomaly_df[(anomaly_df["ticker"]=="SIVB") & anomaly_df["anomaly"]]
frc_flags  = anomaly_df[(anomaly_df["ticker"]=="FRC")  & anomaly_df["anomaly"]]
print("SIGNAL 1: Z-Score Anomaly Detection")
print(f"  SIVB anomaly quarters: {sorted(str(q) for q in sivb_flags['year_quarter'].values)}")
print(f"  FRC  anomaly quarters: {sorted(str(q) for q in frc_flags['year_quarter'].values)}")
print()

# ─── SIGNAL 2: Section Gap ─────────────────────────────────────
sivb_gap = section_quarterly[section_quarterly["ticker"]=="SIVB"].sort_values("year_quarter")
frc_gap  = section_quarterly[section_quarterly["ticker"]=="FRC"].sort_values("year_quarter")
print("SIGNAL 2: Section Gap (MD&A minus Risk Factors)")
print("  SIVB:")
print(sivb_gap[["year_quarter","section_gap"]].round(3).to_string(index=False))
print("  FRC:")
print(frc_gap[["year_quarter","section_gap"]].round(3).to_string(index=False))
print()

# ─── SIGNAL 3: Tier Spread ─────────────────────────────────────
print("SIGNAL 3: Tier-Level MD&A vs Risk Gap")
for tier in ["gsib", "super_regional", "stressed"]:
    tickers_t = [t for t, v in tier_map.items() if v == tier]
    sub = sentiment_df[sentiment_df["ticker"].isin(tickers_t)]
    m = sub.groupby("section")["score"].mean()
    mda_m = m.get("mda", 0); risk_m = m.get("risk_factors", 0)
    print(f"  {tier:20s}  MD&A={mda_m:+.3f}  Risk={risk_m:+.3f}  Gap={mda_m-risk_m:.3f}")
print()

# ─── SIGNAL 4: Survivorship Bias ────────────────────────────────
print("SIGNAL 4: Survivorship Bias")
print(f"  Sector mean with SIVB+FRC: {surv_df['avg_sentiment_all'].mean():.3f}")
print(f"  Sector mean without:       {surv_df['avg_sentiment_survivors'].mean():.3f}")
print(f"  Cohort difference: {surv_df['avg_sentiment_survivors'].mean() - surv_df['avg_sentiment_all'].mean():+.6f} score units")

In [ ]:
print("Synthetic engineering demonstration: no investment recommendation.")
print("Full-series normalisation and outcome-informed generation prevent prospective warning claims.")
print("Empirical work requires authentic dates/sections, recorded model execution and independent evaluation.")


## Export

Schema version 2 labels the cohort difference in score units and records mock backend provenance. The quarterly CSV contains aggregates, not z-scores. The scored CSV contains mock results.


In [ ]:
import json as _json
# Save CSVs
sentiment_df.to_csv(OUTPUTS / "sentiment_scored.csv", index=False)
quarterly.to_csv(OUTPUTS / "quarterly_sentiment.csv", index=False)

# Build run_summary
gsib_tickers = [t for t,v in tier_map.items() if v=="gsib"]
stressed_tickers = [t for t,v in tier_map.items() if v=="stressed"]

def _mean(tickers, section=None):
    sub = sentiment_df[sentiment_df["ticker"].isin(tickers)]
    if section: sub = sub[sub["section"]==section]
    return round(float(sub["score"].mean()), 4)

sivb_flags_list = sorted(str(q) for q in sivb_flags["year_quarter"].values)
frc_flags_list  = sorted(str(q) for q in frc_flags["year_quarter"].values)
max_gap_row = section_quarterly.loc[section_quarterly["section_gap"].idxmax()]
alert_rows = section_quarterly[section_quarterly["section_gap"] > GAP_ALERT]

run_summary = {
    "schema_version": 2,
    "entity_method": "fixed mock frequency table",
    "topic_method": "random mock labels, seed 0",
    "schema_version": 2,
    "entity_method": "fixed mock frequency table",
    "topic_method": "random mock labels, seed 0",
    "scenario": scenario_cfg["name"],
    "n_documents": int(len(corpus)),
    "n_tickers": int(corpus["ticker"].nunique()),
    "n_quarters_analyzed": int(corpus["year_quarter"].nunique()),
    "data_source": "synthetic" if USE_SYNTHETIC else "SEC EDGAR",
    "sentiment_method": "FinBERT" if USE_FINBERT else "mock (correlated)",
    "mock_vs_true_correlation": round(float(np.corrcoef(
        sentiment_df["true_sentiment"], sentiment_df["score"])[0,1]), 4)
        if "true_sentiment" in sentiment_df.columns else None,
    "failed_banks": FAILED_BANKS,
    "failed_banks_last_quarter": "2022Q4",
    "n_anomaly_quarters": int(anomaly_df["anomaly"].sum()),
    "sivb_anomaly_quarters": sivb_flags_list,
    "frc_anomaly_quarters": frc_flags_list,
    "sivb_final_zscore": round(float(
        anomaly_df[anomaly_df["ticker"]=="SIVB"]["z_score"].min()), 4),
    "frc_final_zscore": round(float(
        anomaly_df[anomaly_df["ticker"]=="FRC"]["z_score"].min()), 4),
    "mean_mda_sentiment_gsib": _mean(gsib_tickers, "mda"),
    "mean_risk_sentiment_gsib": _mean(gsib_tickers, "risk_factors"),
    "mean_mda_sentiment_stressed": _mean(stressed_tickers, "mda"),
    "mean_risk_sentiment_stressed": _mean(stressed_tickers, "risk_factors"),
    "max_section_gap_ticker": str(max_gap_row["ticker"]),
    "max_section_gap_quarter": str(max_gap_row["year_quarter"]),
    "max_section_gap_value": round(float(max_gap_row["section_gap"]), 4),
    "gap_alert_quarters_per_ticker": alert_rows["ticker"].value_counts().to_dict(),
    "sector_mean_all": round(float(surv_df["avg_sentiment_all"].mean()), 4),
    "sector_mean_survivors": round(float(surv_df["avg_sentiment_survivors"].mean()), 4),
    "historical_cohort_difference_score": round(float(
        surv_df["avg_sentiment_survivors"].mean() - surv_df["avg_sentiment_all"].mean()), 4),
    "zscore_threshold": ZSCORE_THRESH,
    "section_gap_alert_threshold": GAP_ALERT,
}

(OUTPUTS / "run_summary.json").write_text(_json.dumps(run_summary, indent=2, default=str))
print("Outputs saved:")
for f in sorted(OUTPUTS.iterdir()):
    print(f"  {f.name}")
print()
print(_json.dumps(run_summary, indent=2, default=str))